[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/29_adam.ipynb)

# 🟠 Medium: Adam Optimizer

Implement the **Adam** optimizer from scratch.

### Signature
```python
class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8): ...
    def step(self): ...
    def zero_grad(self): ...
```

### Algorithm (per parameter)
```
m = β1 * m + (1-β1) * grad
v = β2 * v + (1-β2) * grad²
m̂ = m / (1 - β1ᵗ)    # bias correction
v̂ = v / (1 - β2ᵗ)
p -= lr * m̂ / (√v̂ + ε)
```

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch

/usr/local/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [10]:
# ✏️ YOUR IMPLEMENTATION HERE

class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        # store params, init m and v to zeros
        self.t = 0
        self.params = params
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.lr = lr
        self.betas = betas
        self.eps = eps

    def step(self):
        # update params using Adam rule
        self.t += 1
        with torch.no_grad():
             for i in range(len(self.params)):
                self.m[i] = self.betas[0] * self.m[i] + (1 - self.betas[0]) * self.params[i].grad
                self.v[i] = self.betas[1] * self.v[i] + (1 - self.betas[1]) * (self.params[i].grad)**2
                m_correct = self.m[i] / (1 - self.betas[0] ** (self.t))
                v_correct = self.v[i] / (1 - self.betas[1] ** (self.t))
                self.params[i].sub_(self.lr * m_correct / (torch.sqrt(v_correct) + self.eps))
    
    def zero_grad(self):
        for p in self.params:
            p.grad.zero_()

In [11]:
# 🧪 Debug
torch.manual_seed(0)
w = torch.randn(4, 3, requires_grad=True)
opt = MyAdam([w], lr=0.01)
for i in range(5):
    loss = (w ** 2).sum()
    loss.backward()
    opt.step()
    opt.zero_grad()
    print(f'Step {i}: loss={loss.item():.4f}')

Step 0: loss=12.5974
Step 1: loss=12.3944
Step 2: loss=12.1939
Step 3: loss=11.9960
Step 4: loss=11.8005


In [14]:
# ✅ SUBMIT
from torch_judge import check
check('adam')


🧪 Testing: Adam Optimizer (Medium)
──────────────────────────────────────────────────
  ✅ [1/3] Parameters change after step (2.4ms)
  ✅ [2/3] Matches torch.optim.Adam (1012.4ms)
  ✅ [3/3] zero_grad works (0.7ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (1015.5ms total)
  Progress saved. Run status() to see your dashboard.

